<a id="top"></a>

# Lab L1105: Instrument and compare traces

```yaml
title: "Lab L1105: Instrument and compare traces"
keywords: tracing, spans, trajectory, trace diff, signal vs noise, jsonl round-trip, non-determinism, lab
estimated duration: 35
```

> **Lesson:** L11 Tracing. Roadmap: [objectives.md](../../../../docs/origin/lesson_roadmaps/L11/objectives.md) (objectives 3 & 4). Solutions: `L1105_lab_solutions.ipynb`.
>
> Pure Python — no API key needed (scripted FakeModel).

**Learning objectives.** After this lab you can:

- read the spans an instrumented loop emits (a `chain` span, one `llm` span per model call, one `tool` span per dispatch);
- compare two traces of the same task and separate **signal** (a real behavior change) from **noise** (run-to-run jitter);
- round-trip a trace as data (JSONL) and trust it came back unchanged.

The loop is *already* instrumented in `common/agent_loop.py` — you do not rewrite it. You write small functions that **read** the trace it produces.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Trajectory from a trace](#2-problem-1--trajectory-from-a-trace)
- [3. Problem 2 — Write diff_traces](#3-problem-2--write-diff_traces)
- [4. Problem 3 — Signal vs noise (written)](#4-problem-3--signal-vs-noise-written)
- [5. Problem 4 — A trace is data](#5-problem-4--a-trace-is-data)
- [6. Self-check](#6-self-check)

## 1. Setup

We build three runs of the already-instrumented loop, driven by a scripted `FakeModel` so everything is deterministic and keyless:

- **`good`** — a clean two-tool run that terminates naturally.
- **`run_a`** — a runaway: the same failing tool over and over until `max_steps`.
- **`run_b`** — the *same task* as `run_a` after a "tightened prompt", now terminating naturally.

`run_a` vs `run_b` is the before/after-a-prompt-edit A/B pair you'll diff in Problems 2–3.

In [1]:
# Setup: imports + three scripted runs. Runs top-to-bottom, no edits, no API key.
from fluffy_potato_curriculum.common.agent_loop import RunResult, run
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    response,
    text_block,
    tool_use_block,
)
from fluffy_potato_curriculum.common.tools import TOOLS
from fluffy_potato_curriculum.common.tracing import TraceEvent, from_jsonl, to_jsonl

# --- good: a clean two-tool run that terminates naturally ---
good_model = FakeModel(
    [
        response([tool_use_block("c1", "calculator", {"expression": "17*23"})]),
        response([tool_use_block("c2", "lookup", {"city": "Tokyo"})]),
        response([text_block("17 * 23 is 391, and Tokyo has about 37,000,000 people.")]),
    ]
)
good: RunResult = run(good_model, TOOLS, "What is 17 * 23, and what is the population of Tokyo?")

# --- run_a: a runaway. One scripted line, repeated by FakeModel, so the loop keeps
#     asking lookup('Atlantis') until max_steps=4 forces a halt. ---
run_a_model = FakeModel([response([tool_use_block("c1", "lookup", {"city": "Atlantis"})])])
run_a: RunResult = run(run_a_model, TOOLS, "What is the population of Atlantis?", max_steps=4)

# --- run_b: the SAME task after a "tightened prompt". Text only -> natural. ---
run_b_model = FakeModel([response([text_block("I don't have a population on file for Atlantis.")])])
run_b: RunResult = run(run_b_model, TOOLS, "What is the population of Atlantis?")

print("good :", good.termination, good.iterations, "iterations")
print("run_a:", run_a.termination, run_a.iterations, "iterations")
print("run_b:", run_b.termination, run_b.iterations, "iterations")

good : natural 3 iterations
run_a: max_steps 4 iterations
run_b: natural 1 iterations


[↑ Back to top](#top)

## 2. Problem 1 — Trajectory from a trace

The **trajectory** is the ordered list of tool calls the model actually made: each `tool` span's `(name, inputs)`. It's the single most useful thing to pull off a trace — call counts tell you *how much*, but the arguments tell you *what the model was thinking*.

Write `tool_trajectory(trace)` returning a `list[tuple[str, dict]]` of `(name, inputs)` for each `tool` span (skip `llm` and `chain` spans). Then assert the `good` run's trajectory equals the expected sequence.

In [2]:
def tool_trajectory(trace: list[TraceEvent]) -> list[tuple[str, dict[str, object]]]:
    """The ordered (tool_name, args) pairs the model chose, one per tool span."""
    return [(s.name, s.inputs) for s in trace if s.run_type == "tool"]


expected = [
    ("calculator", {"expression": "17*23"}),
    ("lookup", {"city": "Tokyo"}),
]
assert tool_trajectory(good.trace) == expected
print("Problem 1 OK:", tool_trajectory(good.trace))

Problem 1 OK: [('calculator', {'expression': '17*23'}), ('lookup', {'city': 'Tokyo'})]


[↑ Back to top](#top)

## 3. Problem 2 — Write diff_traces

Now compare two traces of the *same task* as data. Write `diff_traces(a, b)` returning a dict with three keys:

- `"trajectory_changed"` — `True` if the `tool_trajectory` (from Problem 1) differs between `a` and `b`;
- `"termination"` — the `(a, b)` pair of termination causes, each read off the `chain` span's `outputs["termination"]`;
- `"total_tokens"` — the `(a, b)` pair of total tokens, each summed over the `llm` spans' `usage.total_tokens`.

Then assert the diff flags `run_a` vs `run_b` as different — the runaway-vs-natural signal.

In [3]:
def termination_of(trace: list[TraceEvent]) -> str:
    """The termination cause, read off the chain span's outputs."""
    chain = next(s for s in trace if s.run_type == "chain")
    return str(chain.outputs["termination"])


def total_tokens(trace: list[TraceEvent]) -> int:
    """Total tokens across all llm spans."""
    return sum(s.usage.total_tokens for s in trace if s.usage is not None)


def diff_traces(a: list[TraceEvent], b: list[TraceEvent]) -> dict[str, object]:
    """Compare two traces of the same task on trajectory, termination, and tokens.

    Example output:
        {"trajectory_changed": True,
         "termination": ("max_steps", "natural"),
         "total_tokens": (480, 120)}
    """
    return {
        "trajectory_changed": tool_trajectory(a) != tool_trajectory(b),
        "termination": (termination_of(a), termination_of(b)),
        "total_tokens": (total_tokens(a), total_tokens(b)),
    }


diff = diff_traces(run_a.trace, run_b.trace)
print(diff)
assert diff["trajectory_changed"] is True
assert diff["termination"] == ("max_steps", "natural")
print("Problem 2 OK: the diff flags run_a vs run_b as different")

{'trajectory_changed': True, 'termination': ('max_steps', 'natural'), 'total_tokens': (480, 120)}
Problem 2 OK: the diff flags run_a vs run_b as different


[↑ Back to top](#top)

## 4. Problem 3 — Signal vs noise (written)

Run the cell below to print the diff again, then answer in the markdown cell that follows.

The diff reports three differences between `run_a` and `run_b`. In a sentence or two each:

1. Which difference is **signal** (a real behavior change you'd act on)?
2. Which would be **noise** (run-to-run variation that means nothing on its own)?
3. Why can a *single* `run_a`-vs-`run_b` comparison not, by itself, prove the prompt edit fixed the runaway?

In [4]:
# Apply your diff_traces to the A/B pair and read off the three differences.
for key, value in diff_traces(run_a.trace, run_b.trace).items():
    print(f"{key:20} {value}")

trajectory_changed   True
termination          ('max_steps', 'natural')
total_tokens         (480, 120)


_Write your answer by editing this cell (double-click)._

1. **Signal:** the trajectory + termination change. `run_a` made four failing `lookup('Atlantis')` calls and ended in `max_steps`; `run_b` made zero tool calls and ended `natural`. That is a real behavioral change — the runaway loop is gone — and exactly the kind of fix you'd act on.

2. **Noise:** the `total_tokens` difference on its own. Token counts (and latency) jitter from run to run even when behavior is identical; here they only moved *because* the trajectory changed. A token delta by itself proves nothing.

3. **Why one comparison can't prove the fix:** the loop is non-deterministic, so `run_a` could have been an unlucky path and `run_b` a lucky one. A single before/after pair is suggestive, not conclusive — you'd need the same task across *several* runs to trust that the prompt edit, not luck, stopped the runaway. (That need for many repeatable cases is the seed of L12's eval set.)

[↑ Back to top](#top)

## 5. Problem 4 — A trace is data

A trace is just a list of typed records, so it serializes to JSON-lines and parses straight back unchanged. That round-trip is what makes a trace *durable* — you can store it and read it later instead of re-running a non-deterministic agent.

Round-trip the `good` trace: serialize it with `to_jsonl`, parse it back with `from_jsonl`, and assert the recovered spans equal the original. (`TraceEvent` is a Pydantic model, so `==` compares by value.)

In [5]:
# Serialize the good trace to JSONL, parse it back, and confirm it's unchanged.
jsonl = to_jsonl(good.trace)
recovered = from_jsonl(jsonl)

print(f"{len(jsonl.splitlines())} JSONL lines -> {len(recovered)} spans recovered")
assert recovered == good.trace
print("Problem 4 OK: the trace round-tripped through JSONL unchanged")

6 JSONL lines -> 6 spans recovered
Problem 4 OK: the trace round-tripped through JSONL unchanged


[↑ Back to top](#top)

## 6. Self-check

Compare against `L1105_lab_solutions.ipynb`. You're done when:

- Problem 1: `tool_trajectory(good.trace)` equals `[("calculator", {"expression": "17*23"}), ("lookup", {"city": "Tokyo"})]`.
- Problem 2: `diff_traces(run_a.trace, run_b.trace)` has `trajectory_changed=True` and `termination=("max_steps", "natural")`.
- Problem 3: your written answer names trajectory/termination as signal, tokens as noise, and explains why one run can't prove a fix.
- Problem 4: the `good` trace round-trips through `to_jsonl` / `from_jsonl` unchanged.

[↑ Back to top](#top)